# Multi-Benchmark Evaluation — 8 Benchmarks, $0 Cost

Extends the study from GSM8K to **8 diverse benchmarks** across math and visual reasoning.

## Two Evaluation Protocols

### Protocol A — Text-based math (render text as images)
| Benchmark | Problems | Type | What it tests |
|-----------|----------|------|---------------|
| GSM8K | 1,319 | Arithmetic word problems | Grade-school math |
| SVAMP | 1,000 | Variation-resistant math | Robustness to rewording |
| AQuA-RAT | 254 | Algebraic (multiple choice) | Algebra + MC format |
| MATH | 5,000 | Competition-level | Hard math, LaTeX |

### Protocol B — Visual benchmarks (native images)
| Benchmark | Problems | Type | What it tests |
|-----------|----------|------|---------------|
| MathVista | 1,000 | Visual math | Charts, geometry, diagrams |
| ScienceQA | ~4,000 | Science MC | Diagrams + knowledge |
| AI2D | 4,903 | Diagram QA | Scientific diagrams |
| ChartQA | 2,500 | Chart QA | Bar/line/pie charts |

**Key Question:** Does the modality gap found in GSM8K generalize to other tasks?

---

### Strategy
Pick 1-2 benchmarks per session alongside 1-2 models. Mix Protocol A and B benchmarks.

**Recommended session plan:**
- Session A1: GSM8K + SVAMP on Model 0,1 (~5h)
- Session A2: AQuA-RAT + MATH on Model 0,1 (~4h)
- Session B1: MathVista + ChartQA on Model 0,1 (~5h)
- Session B2: ScienceQA + AI2D on Model 0,1 (~5h)
- Repeat for additional models...

In [ ]:
!pip install torch transformers datasets scipy pyyaml tqdm pillow bitsandbytes torchvision --quiet

In [ ]:
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/content/repo')

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/vlm_research_results'
!mkdir -p {OUTPUT_DIR}

In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURATION — CHANGE EACH SESSION
# ══════════════════════════════════════════════════════════

# Which benchmarks to run THIS session
BENCHMARKS = ['gsm8k', 'svamp']  # ◀── CHANGE THIS

# Which model to run THIS session
MODEL = {
    'name': 'Qwen/Qwen2-VL-2B-Instruct',
    'type': 'qwen',
    'max_new_tokens': 256,
    'torch_dtype': 'bfloat16',
    'quantize': None,
}

NUM_PROBLEMS = None  # None = full dataset, or integer for quick test

# Available benchmarks:
from src.benchmarks import list_benchmarks
list_benchmarks()

In [ ]:
import os, time, json, torch
import pandas as pd
from collections import Counter

from src.models import VLMModel
from src.benchmarks import load_benchmark, get_benchmark_info
from src.benchmark_eval import (
    run_protocol_a, run_protocol_b,
    save_protocol_a_results, save_protocol_b_results,
)

torch.manual_seed(42)

# Load model
vlm = VLMModel(**MODEL)
vlm.load()
short = MODEL['name'].split('/')[-1]

session_stats = {}

for bench_name in BENCHMARKS:
    info = get_benchmark_info(bench_name)
    
    # Skip if done
    stats_path = os.path.join(OUTPUT_DIR, short, bench_name, 'statistics.json')
    if os.path.exists(stats_path):
        print(f'\n>>> SKIP {bench_name} — already done <<<')
        with open(stats_path) as f:
            session_stats[bench_name] = json.load(f)
        continue
    
    print(f"\n{'='*60}")
    print(f'  BENCHMARK: {bench_name} (protocol={"A" if not info["has_images"] else "B"})')
    print(f"{'='*60}")
    
    t0 = time.time()
    items = load_benchmark(bench_name, NUM_PROBLEMS)
    
    if not info['has_images']:
        # Protocol A: text-based
        image_dir = os.path.join(OUTPUT_DIR, 'images', bench_name)
        results = run_protocol_a(vlm, items, bench_name, OUTPUT_DIR, image_dir)
        stats = save_protocol_a_results(results, items, MODEL['name'], bench_name, OUTPUT_DIR)
    else:
        # Protocol B: visual
        results, eval_items = run_protocol_b(vlm, items, bench_name, OUTPUT_DIR)
        stats = save_protocol_b_results(results, eval_items, MODEL['name'], bench_name, OUTPUT_DIR)
    
    elapsed = time.time() - t0
    print(f'\nTime: {elapsed/60:.1f} min')
    session_stats[bench_name] = stats

vlm.unload()
print('\n\nSession complete!')

In [ ]:
# ── Summary of this session ──
print(f'\nResults for {short}:')
print('-' * 70)
for bench, stats in session_stats.items():
    if 'acc_text' in stats:
        print(f"  {bench:12s}  Text={stats['acc_text']:.3f}  "
              f"Image={stats['acc_img']:.3f}  "
              f"Drop={stats['acc_drop']:+.3f}  "
              f"p={stats['mcnemar_p']:.4f}")
    elif 'acc_multimodal' in stats:
        print(f"  {bench:12s}  MM={stats['acc_multimodal']:.3f}  "
              f"Text={stats['acc_text_only']:.3f}  "
              f"Diff={stats['acc_diff']:+.3f}  "
              f"p={stats['mcnemar_p']:.4f}")

## Cross-Benchmark Aggregation (run after all sessions complete)

In [ ]:
# ── Aggregate ALL results from Drive ──
import os, json
import pandas as pd

all_results = {}
for model_dir in os.listdir(OUTPUT_DIR):
    model_path = os.path.join(OUTPUT_DIR, model_dir)
    if not os.path.isdir(model_path) or model_dir in ['images', 'rendered_images']:
        continue
    for bench_dir in os.listdir(model_path):
        stats_file = os.path.join(model_path, bench_dir, 'statistics.json')
        if os.path.isfile(stats_file):
            with open(stats_file) as f:
                stats = json.load(f)
            if model_dir not in all_results:
                all_results[model_dir] = {}
            all_results[model_dir][bench_dir] = stats

# Build summary table
rows = []
for model, benchmarks in sorted(all_results.items()):
    for bench, stats in sorted(benchmarks.items()):
        row = {'Model': model, 'Benchmark': bench, 'N': stats.get('n', '?')}
        if 'acc_text' in stats:  # Protocol A
            row['Protocol'] = 'A'
            row['Acc_1'] = f"{stats['acc_text']:.3f}"
            row['Acc_2'] = f"{stats['acc_img']:.3f}"
            row['Label_1'] = 'Text'
            row['Label_2'] = 'Image'
            row['Drop'] = f"{stats['acc_drop']:+.3f}"
        else:  # Protocol B
            row['Protocol'] = 'B'
            row['Acc_1'] = f"{stats['acc_multimodal']:.3f}"
            row['Acc_2'] = f"{stats['acc_text_only']:.3f}"
            row['Label_1'] = 'Multimodal'
            row['Label_2'] = 'Text-Only'
            row['Drop'] = f"{stats['acc_diff']:+.3f}"
        row['p-value'] = f"{stats['mcnemar_p']:.2e}"
        sig = stats['mcnemar_p']
        row['Sig'] = '***' if sig<0.001 else '**' if sig<0.01 else '*' if sig<0.05 else 'ns'
        row["Cohen's h"] = f"{stats.get('cohens_h', 0):.3f}"
        rows.append(row)

summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, 'full_benchmark_summary.csv'), index=False)
print(f"\nSaved to {OUTPUT_DIR}/full_benchmark_summary.csv")

In [ ]:
# ── Heatmap: modality gap across models × benchmarks ──
import matplotlib.pyplot as plt
import numpy as np

models = sorted(all_results.keys())
benchmarks = sorted(set(b for m in all_results.values() for b in m.keys()))

matrix = np.full((len(models), len(benchmarks)), np.nan)
for i, m in enumerate(models):
    for j, b in enumerate(benchmarks):
        if b in all_results.get(m, {}):
            s = all_results[m][b]
            drop = s.get('acc_drop', s.get('acc_diff', 0))
            matrix[i, j] = drop * 100  # convert to pp

fig, ax = plt.subplots(figsize=(max(10, len(benchmarks)*1.5), max(4, len(models)*0.6)))
im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=-40, vmax=40)
ax.set_xticks(range(len(benchmarks)))
ax.set_xticklabels(benchmarks, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=10)

# Annotate cells
for i in range(len(models)):
    for j in range(len(benchmarks)):
        if not np.isnan(matrix[i,j]):
            ax.text(j, i, f'{matrix[i,j]:+.1f}', ha='center', va='center',
                    fontsize=9, fontweight='bold',
                    color='white' if abs(matrix[i,j]) > 20 else 'black')

plt.colorbar(im, label='Accuracy Change (pp)', shrink=0.8)
ax.set_title('Modality Gap Across Models × Benchmarks (pp)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'modality_gap_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved!')